In [143]:
import os
import pathlib as path
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from nilearn.datasets import load_mni152_template
from nilearn.image import load_img, resample_to_img, concat_imgs, math_img
from nilearn.masking import apply_mask, compute_brain_mask, unmask
from nilearn.plotting import plot_stat_map, plot_glass_brain
from scipy.stats import ttest_1samp
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [ ]:
# Beta series correlation
proj_path = "/mnt/1e99f03a-239b-4885-af31-60eb1322a5e1/IEL/sklearn"

def part(subj):
    proj = 'IEL'
    subj = f"{subj:03d}"
    part_id = f"{proj}{subj}"
    return part_id

def input_metadata_path(subj):
    part_id = part(subj)
    metadata_path = os.path.join(proj_path,'data','individual',part_id,'Metadata',f'{part_id}_dataset.csv')
    return metadata_path

def output_metadata_path(subj):
    part_id = part(subj)
    output_dir = os.path.join(proj_path,'data','individual',part_id,'FC','ROI')
    return output_dir

conditions = ['Cue_Loss_Neutral', 'Cue_NoLoss_Neutral', 'Cue_Loss_Positive','Cue_NoLoss_Positive']
     
def beta_data_path(roi):
    for condition in conditions:
        return os.path.join(proj_path, 'data','individual',part_id, 'BetaSeries', condition)

roi_list = {'LeftAmygdala' : 'HarvardOxford-sub-maxprob-thr50-2mm_LAmyg_F.nii.gz',
            'RightAmygdala' : 'HarvardOxford-sub-maxprob-thr50-2mm_RAmyg_F.nii.gz',
            'Nucleus Accumbens' : 'HarvardOxford-sub-maxprob-thr50-3mm_BothNAcc_F.nii.gz'}
            # 'VentraLStriatum' : 'HarvardOxford-sub-maxprob-thr50-1mm-BothVS_F.nii.gz',
            # 'Putamen' : 'HarvardOxford-sub-maxprob-thr50-1mm-BothPutamen_F.nii.gz',
            # 'SN-VTA' : 'Pauli2018_50percentprob_bin_1mm_BothSN_VTA_F.nii.gz',
            # 'VmPFC' : 'vmpfc_Lindquist2015_F.nii.gz',
            # 'Insula' : 'BothAnteriorInsula_Mask_F.nii.gz',
            # 'PHG' : 'visfAtlas_MNI152_volume_PPA_F_binary.nii.gz'}

temp_path = "/media/cogemolab/home2/templates"
 
def roi_template_path(roi_name):
    roi_path = os.path.join(temp_path,roi_list[roi_name])
    return roi_path

group_matrix = []
group_seed_matrix = []

# subject loop
for subj in range(1,2):

    #part_id
    part_id = part(subj)

    #metadata file
    metadata_file = pd.read_csv(input_metadata_path(subj))

    roi_matrix = {}

    #loop for ROI
    for roi in roi_list.keys():
        
        # ROI dataframe used later    
        roi_df = metadata_file.copy()

        #ROI mask path
        mask_path = roi_template_path(roi)
        mask_img = load_img(mask_path)

        # loop foe each trial
        for index, rows in metadata_file.iterrows():
          
            #beta path
            beta_path = rows['Beta_Path']
            
            #load beta file
            beta_img = load_img(beta_path)

            #resampling roi mask if the beta file
            if beta_img.shape != mask_img.shape:
                resampled_mask_img = resample_to_img(mask_img,beta_img, interpolation='nearest')
            else:
                resampled_mask_img = mask_img

            # Convert all values greater than 0 to 1
            binary_mask_img = math_img("img > 0", img=resampled_mask_img)

            #extracting roi betas
            voxel_betas_roi = apply_mask(beta_img,binary_mask_img)

            #voxels betas
            for v in range(len(voxel_betas_roi)):
                roi_df.loc[index, f'voxel_{v+1}'] = voxel_betas_roi[v]


        #  extract the mean voxel value per trial
        voxel_col = roi_df.columns[roi_df.columns.str.startswith('voxel_')]
        roi_df['voxel_mean'] = np.mean(roi_df[voxel_col], axis=1)

        roi_matrix[roi] = roi_df['voxel_mean']

    all_roi_mean = pd.DataFrame(roi_matrix)

    print(all_roi_mean)

    #Save the ROI matrix dataframe
    output_dir = output_metadata_path(subj)
    os.makedirs(output_dir,exist_ok = True)
    roi_output_file = os.path.join(output_dir,f"{part_id}_ROI_Matrix.csv")
    all_roi_mean.to_csv(roi_output_file,sep="\t",index=False)

    # ROI-ROI Correlation matrix
    correlation_matrix = all_roi_mean.corr().values
    z_correlation_matrix = 0.5 * np.log((1 + correlation_matrix) / (1 - correlation_matrix))
    group_matrix.append(z_correlation_matrix)

    plt.figure(figsize=(5,3))
    sns.heatmap(z_correlation_matrix,  annot=True, cmap='coolwarm', fmt=".2f")
    plt.title('Correlation Matrix')
    plt.show()
    # Condition wise correlation matrix
    condition = pd.DataFrame()
    all_roi_mean['Emotion'] = metadata_file['Emotion']
    all_roi_mean['Loss'] = metadata_file['Loss']
    for emotion in all_roi_mean['Emotion'].unique():
            for loss in all_roi_mean['Loss'].unique():
                condition = all_roi_mean[(all_roi_mean['Emotion']==emotion)
                                            & (all_roi_mean['Loss']==loss)]

                roi_corr = condition[list(roi_list.keys())].corr()
                plt.figure(figsize=(5,4))

                sns.heatmap(roi_corr,
                    annot=True,
                    cmap='coolwarm',
                    vmin=-1,
                    vmax=1,
                    fmt=".2f")

                plt.title(f"{loss}_{emotion}")
                plt.show()

    ##### Seed based correlation #####
    seed_roi = 'LeftAmygdala'

    # Extract voxel betas across all trials
    img_list = []
    for beta in metadata_file['Beta_Path']:
        beta_img = load_img(beta)
        img_list.append(beta_img)
    beta_xyztrial = concat_imgs(img_list)

    # Loading brain mask - MNI template
    brain_mask = compute_brain_mask(beta_xyztrial)

    # Creating voxel matrix
    voxel_matrix = apply_mask(beta_xyztrial,brain_mask)
    print(voxel_matrix.shape)

    for emotion in all_roi_mean['Emotion'].unique():
        for loss in all_roi_mean['Loss'].unique():

            condition = all_roi_mean[(all_roi_mean['Emotion'] == emotion)&(all_roi_mean['Loss'] == loss)]

            seed_series = condition[seed_roi].values
            voxel_matrix_cond = voxel_matrix[condition.index]

            seed_minus_mean = (seed_series- np.mean(seed_series))

            voxel_minus_mean = (voxel_matrix_cond- np.mean(voxel_matrix_cond, axis=0))

            correlation_map = (np.dot(seed_minus_mean, voxel_minus_mean)/
                (np.sqrt(np.sum(seed_minus_mean**2))*np.sqrt(np.sum(voxel_minus_mean**2,axis=0))))

            # Fisher z transform
            z_map = 0.5 * np.log((1 + correlation_map) / (1 - correlation_map))
            group_seed_matrix.append(z_map)
            FC_brain_map_z = unmask(z_map, brain_mask)                
            gm_mask = load_mni152_template()

            # Visualisation of conditon wise FC connectivity between seed and whole brain voxels
            plot_stat_map(FC_brain_map_z,
                gm_mask,
                display_mode='ortho',
                threshold=0.7,
                black_bg=False,
                draw_cross=False,
                colorbar=True,
                cmap='cold_hot',
                title=f'{seed_roi} | {loss}_{emotion}',
                dim=-0.3)

# # Group Analyses ROI-ROI
group_matrix = np.array(group_matrix)  # (subjects, ROIs, ROIs)

# sanity check
print("Group matrix shape:", group_matrix.shape)

t_map, p_map = ttest_1samp(group_matrix, 0, axis=0)

print("T-map:\n", t_map)
print("P-map:\n", p_map)

# # Group Analyses Seed based
group_seed_matrix = np.array(group_seed_matrix)
t_map, p_map = ttest_1samp(group_seed_matrix,0,axis=0)

print("T-map_seedbased:\n", t_map)
print("P-map_seedbased:\n", p_map)

gm_mask = load_mni152_template()
FC_t_img = unmask(t_map, brain_mask)
plot_stat_map(FC_t_img,gm_mask,
              colorbar = True,
              cmap='cold_hot',
              title=f'{seed_roi} | {loss}_{emotion}',
              dim=-0.3)   
